In [4]:
import json
import time
import os
import random
import hexbytes
from hexbytes import HexBytes
from pathlib import Path
import requests
from requests.exceptions import HTTPError
from datetime import datetime, timezone
from web3 import Web3
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
import concurrent.futures
import sys
import threading
from functools import wraps

# === CONFIGURATION ===
INFURA_URL = "https://mainnet.infura.io/v3/3921fc62a7ce4cda98926f47409b3d19"
ETHERSCAN_API_KEY = "F7K9BTHSSB9EQT9WEGHMG3VFJ54KA8RM1K"

CONTRACT_ADDRESS = POOL_ADDRESS = "0xCBCdF9626bC03E24f779434178A73a0B4bad62eD"
ABI_FILE = "WETH_WBTC_pool.json"  # Load your contract ABI file
BLOCKS_FILE = "blocks_data.json"
TRANSACTIONS_FILE = "transactions.json"
METADATA_FILE = "processed_blocks.json"
BATCH_SIZE = 1000  # Number of transactions to process before writing to disk

# === CONNECT TO ETHEREUM NODE ===
# w3 = Web3(Web3.HTTPProvider(INFURA_URL))
w3 = Web3(
    Web3.HTTPProvider("https://mainnet.infura.io/v3/3921fc62a7ce4cda98926f47409b3d19")
    #Web3.HTTPProvider("http://127.0.0.1:8545")
)
assert w3.is_connected(), "Web3 provider connection failed"

In [5]:
w3.eth.get_block('latest').number

23361818

In [6]:
# --------------------
# Helper Function: Get ABI from Etherscan or Disk
# --------------------

def get_abi(contract_address: str, api_key: str) -> list:
    """
    Retrieves the ABI for a given contract address.
    Checks if the ABI is available in the local 'ABI' folder.
    If not, it fetches the ABI from Etherscan using the provided API key,
    then saves it to disk for future use.
    
    Parameters:
        contract_address (str): The contract address (checksum not required here).
        api_key (str): Your Etherscan API key.
        
    Returns:
        list: The ABI loaded as a Python list.
    """
    # Ensure the ABI folder exists.
    abi_folder = "ABI"
    if not os.path.exists(abi_folder):
        os.makedirs(abi_folder)
    
    # Save ABI with filename based on contract address.
    filename = os.path.join(abi_folder, f"{contract_address}.json")
    
    # If file exists, load and return the ABI.
    if os.path.exists(filename):
        with open(filename, "r") as file:
            abi = json.load(file)
    else:
        # Construct the Etherscan API URL.
        url = f"https://api.etherscan.io/api?module=contract&action=getabi&address={contract_address}&apikey={api_key}"
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            if data["status"] == "1":
                # Parse the ABI and save it for later use.
                abi = json.loads(data["result"])
                with open(filename, "w") as file:
                    json.dump(abi, file)
            else:
                raise Exception(f"Error fetching ABI for contract {contract_address}: {data['result']}")
        else:
            raise Exception("Error connecting to the Etherscan API. Status code: " + str(response.status_code))
    return abi

# -----------------------
# Helper: Convert event to dict
# -----------------------
def event_to_dict(event):
    d = dict(event)
    if "args" in d:
        d["args"] = dict(d["args"])
    if "transactionHash" in d:
        d["transactionHash"] = d["transactionHash"].hex()
    if "blockHash" in d:
        d["blockHash"] = d["blockHash"].hex()
    return d

# -----------------------
# Metadata Computation from Events File
# -----------------------
def load_metadata_from_events():
    """
    Load metadata directly from EVENTS_FILE.
    Returns a dict with keys as chunk keys (e.g. "0-9999") and values as dicts of event type counts.
    """
    metadata = {}
    try:
        with open(EVENTS_FILE, "r") as f:
            for line in f:
                if line.strip():
                    try:
                        event = json.loads(line)
                        block_number = int(event.get("blockNumber", 0))
                        event_type = event.get("event", "Unknown")
                        chunk_start = (block_number // CHUNK_SIZE) * CHUNK_SIZE
                        chunk_end = chunk_start + CHUNK_SIZE - 1
                        chunk_key = f"{chunk_start}-{chunk_end}"
                        if chunk_key not in metadata:
                            metadata[chunk_key] = {}
                        metadata[chunk_key][event_type] = (
                            metadata[chunk_key].get(event_type, 0) + 1
                        )
                    except Exception as e:
                        print(f"Error processing a line in events file: {e}")
                        continue
    except FileNotFoundError:
        pass
    return metadata


def get_contract_creation_block_etherscan(contract_address: str, etherscan_api_key: str) -> int:
    """
    Retrieves the contract creation block from Etherscan.
    Returns the block number as an integer.
    """
    url = (f"https://api.etherscan.io/api?module=contract&action=getcontractcreation"
           f"&contractaddresses={contract_address}&apikey={etherscan_api_key}")
    response = requests.get(url)
    data = response.json()

    if data.get("status") == "1":
        results = data.get("result", [])
        if results and len(results) > 0:
            return int(results[0]["blockNumber"])
        else:
            raise Exception("No contract creation data found.")
    else:
        raise Exception("Error fetching creation block: " + data.get("result", "Unknown error"))


# Used to find at which block 1 contract has been deployed
# Might be useful later, put it in JSON in the end
def get_contract_creation_block_custom(start_block=0, end_block=100000):

    def get_contract_deployments(start_block, end_block, max_workers=8):
        deployments = []

        def process_block(block_number):
            block = w3.eth.get_block(block_number, full_transactions=True)
            block_deployments = []
            for tx in block.transactions:
                if tx.to is None:
                    try:
                        receipt = w3.eth.get_transaction_receipt(tx.hash)
                        contract_address = receipt.contractAddress
                        if contract_address:
                            block_deployments.append(
                                {
                                    "block_number": block_number,
                                    "contract_address": contract_address,
                                }
                            )
                    except:
                        print(tx.hash)
            return block_deployments

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            future_to_block = {
                executor.submit(process_block, bn): bn
                for bn in range(start_block, end_block + 1)
            }
            for future in as_completed(future_to_block):
                block_deployments = future.result()
                deployments.extend(block_deployments)

        return deployments

    deployments = get_contract_deployments(start_block, end_block)

    # Save the results to a JSON file
    with open("contract_deployments.json", "w") as f:
        json.dump(deployments, f, indent=4)


# -- Step 2: Reconstruct an Event’s Signature --
def get_event_signature(event_name: str, abi: list) -> str:
    """
    Given an event name and an ABI, find the event definition and reconstruct its signature.
    For example, for event Transfer(address,address,uint256) this returns its keccak256 hash.
    """
    from eth_utils import keccak, encode_hex

    for item in abi:
        if item.get("type") == "event" and item.get("name") == event_name:
            # Build the signature string: "Transfer(address,address,uint256)"
            types = ",".join([inp["type"] for inp in item.get("inputs", [])])
            signature = f"{event_name}({types})"
            return encode_hex(keccak(text=signature))
    raise ValueError(f"Event {event_name} not found in ABI.")


def block_to_utc(block_number):
    """
    Convert a block number into its UTC timestamp.

    Parameters:
        w3 (Web3): A Web3 instance
        block_number (int): The block number

    Returns:
        datetime: The block timestamp in UTC
    """
    block = w3.eth.get_block(block_number)
    timestamp = block["timestamp"]
    return datetime.fromtimestamp(timestamp, tz=timezone.utc).isoformat()


# token_address = w3.to_checksum_address("0x86c8bF8532AA2601151c9DbbF4e4C4804e042571")
# token_abi = get_abi(token_address, ETHERSCAN_API_KEY)
# token_contract = w3.eth.contract(address=token_address, abi=token_abi)

token_address = w3.to_checksum_address("0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2")
token_abi = get_abi(token_address, ETHERSCAN_API_KEY)
token_contract = w3.eth.contract(address=token_address, abi=token_abi)

In [5]:
### We focus on the Factory contract of uniswap v1 "0xc0a47dFe034B400B47bDaD5FecDa2621de6c4d95"
# We need the ProviderNode to be initialized already
# Why debug_traceTransaction Is the Best Option
# It replays the transaction within the exact historical state using your archive node and returns a detailed call graph, including internal calls and value flows—not just high-level transfers. You can choose tracers like callTracer (which outputs call frames and nested structure) for highest clarity and insight.
# Alternatives like event logs or transaction receipts won't capture internal calls, since those are not emitted as events. You need a trace API to follow what's happening inside smart contract execution.
uniswap_v1_factory_address = w3.to_checksum_address("0xc0a47dFe034B400B47bDaD5FecDa2621de6c4d95")
uniswap_v1_factory_abi = get_abi(uniswap_v1_factory_address, ETHERSCAN_API_KEY)
uniswap_v1_factory_contract = w3.eth.contract(
    address=uniswap_v1_factory_address, abi=uniswap_v1_factory_abi
)

def trace_internal_transactions(tx_hash: str, tracer: str = "callTracer") -> dict:
    """
    Performs debug_traceTransaction with specified tracer (default: callTracer).
    Returns the full trace result as a Python dict.
    """
    trace = w3.provider.make_request(
        "debug_traceTransaction", [tx_hash, {"tracer": tracer}]
    )
    return trace.get("result", {})


def extract_internal_transfers_from_trace(trace: dict) -> list:
    """
    Recursively traverses the 'calls' in the trace to gather internal transfers.
    Returns list of dicts with from, to, value, gasUsed, etc.
    """
    transfers = []

    def recurse(call):
        # Internal transfer if value is non-zero
        value = int(call.get("value", "0x0"), 16)
        if value > 0:
            transfers.append(
                {
                    "from": call.get("from"),
                    "to": call.get("to"),
                    "value": value,
                    "gas": int(call.get("gas", "0x0"), 16),
                    "gasUsed": int(call.get("gasUsed", "0x0"), 16),
                    "type": call.get("type"),
                    "error": call.get("error"),
                }
            )
        for sub in call.get("calls", []) or []:
            recurse(sub)

    recurse(trace)
    return transfers


def get_internal_transactions_for_contract(
    contract_address: str, from_block: int, to_block: int
):
    """Scan blocks, identify txs to/from contract, and trace internal calls."""
    results = []
    for block_num in range(from_block, to_block + 1):
        block = w3.eth.get_block(block_num, full_transactions=True)
        for tx in block.transactions:
            if (
                 tx["to"]
                 and tx["to"] == uniswap_v1_factory_address
                 or tx["from"] == uniswap_v1_factory_address
            ):

                function, params = uniswap_v1_factory_contract.decode_function_input(
                    tx["input"]
                )
                if function.fn_name == 'createExchange':
                    #print(tx)
                    # print('Called function:', function.fn_name)
                    #print('With arguments:', params)
                    uni_created_token = params['token']
                    univ1_token_address = w3.to_checksum_address(uni_created_token)
                    univ1_factory_abi = get_abi(univ1_token_address, ETHERSCAN_API_KEY)
                    univ1_factory_contract = w3.eth.contract(
                        address=univ1_token_address, abi=univ1_factory_abi
                    )
                    token_name =  None
                    token_symbol = None 
                    try:
                        token_name = univ1_factory_contract.functions.name().call()
                        token_symbol = univ1_factory_contract.functions.symbol().call()
                        print(f"Token Name: {token_name}")
                        print(f"Token Symbol: {token_symbol}")
                    except:
                        print(f"Contract is a proxy {uni_created_token}")

                    token_created_exchange_address = uniswap_v1_factory_contract.functions.getExchange(
                        uni_created_token
                    ).call()
                    print(f"Token {token_name} UniExchange_address: {token_created_exchange_address}")
                    a = w3.to_checksum_address(token_created_exchange_address)
                    b = get_abi(a, ETHERSCAN_API_KEY)
                    c = w3.eth.contract(
                        address=a, abi=b
                    )
                    try:
                        #print(f"{c.functions.name.call()}")
                        #print(f"{c.functions.symbol.call()}")
                        print(f"Token Address: {c.functions.tokenAddress.call()}")
                        #print(
                        #    f"Is this the same ? {c.functions.tokenAddress.call() == uni_created_token}"
                        #)
                    except:
                        print(f"proxy: {c}")
            #     tx_hash = tx.hash.hex()
            #     trace = trace_internal_transactions(tx_hash)
            #     transfers = extract_internal_transfers_from_trace(trace)
            #     results.append(
            #         {
            #             "tx_hash": tx_hash,
            #             "block": block_num,
            #             "internal_transfers": transfers,
            #         }
            #     )
    return results

internal_txs = get_internal_transactions_for_contract(
    uniswap_v1_factory_contract, 6627900, w3.eth.get_block("latest").number
    #uniswap_v1_factory_contract, 6500000, w3.eth.get_block("latest").number
)
for entry in internal_txs:
    print(entry["tx_hash"], entry["internal_transfers"])

Token Name: b'Maker\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'
Token Symbol: b'MKR\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'
Token b'Maker\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00' UniExchange_address: 0x2C4Bd064b998838076fa341A83d007FC2FA50957
Token Address: 0x9f8F72aA9304c8B593d555F12eF6589cC3A579A2
Token Name: OMGToken
Token Symbol: OMG
Token OMGToken UniExchange_address: 0xDdee242662323a3cff3F9AA139fFA496aC3C73B0
Token Address: 0xd26114cd6EE289AccF82350c8d8487fedB8A0C07
Token Name: 0x Protocol Token
Token Symbol: ZRX
Token 0x Protocol Token UniExchange_address: 0xaE76c84C9262Cdb9abc0C2c8888e62Db8E22A0bF
Token Address: 0xE41d2489571d322189246DaFA5ebDe1F4699F498
Token Name: BNB
Token Symbol: BNB
Token BNB UniExchange_address: 0x255e60c9d597dCAA66006A904eD36424F7B26286
Token Addre

KeyboardInterrupt: 

In [ ]:
# Once having the uniswap exchange for 1 token, we need to analyze the liquidity movement, swap, volume and fees
# BNB: 0x255e60c9d597dCAA66006A904eD36424F7B26286
uniswap_v1_BNB_exchange_address = w3.to_checksum_address(
    "0x255e60c9d597dCAA66006A904eD36424F7B26286"
    # "0x2157A7894439191E520825FE9399AB8655E0F708"
)
uniswap_v1_BNB_exchange_abi = get_abi(uniswap_v1_BNB_exchange_address, ETHERSCAN_API_KEY)
uniswap_v1_BNB_exchange_contract = w3.eth.contract(
    address=uniswap_v1_BNB_exchange_address, abi=uniswap_v1_BNB_exchange_abi
)
# from_block = 6827900
from_block = 6800000
step = 100000

def extract_liquidity_data(logs, provider_address, start_block, end_block):
    """Extract and format liquidity data for a specific provider."""
    liquidity_data = defaultdict(int)

    for log in logs:
        log = parse_event(log)
        if log["blockNumber"] < start_block or log["blockNumber"] > end_block:
            continue

        if log["args"]["provider"].lower() == provider_address.lower():
            if log["event"] == "AddLiquidity":
                liquidity_data[log["blockNumber"]] += (
                    log["args"]["eth_amount"] + log["args"]["token_amount"]
                )
            elif log["event"] == "RemoveLiquidity":
                liquidity_data[log["blockNumber"]] -= (
                    log["args"]["eth_amount"] + log["args"]["token_amount"]
                )
            elif log["event"] == "Transfer":
                if log["args"]["_from"].lower() == provider_address.lower():
                    liquidity_data[log["blockNumber"]] -= log["args"]["_value"]
                elif log["args"]["_to"].lower() == provider_address.lower():
                    liquidity_data[log["blockNumber"]] += log["args"]["_value"]

    return dict(liquidity_data)


for i in range(from_block, w3.eth.get_block('latest').number, step):
    # for event in uniswap_v1_BNB_exchange_contract.events:
    # event is now the class/type of the event, e.g. 'Swap'
    # event_fn = getattr(uniswap_v1_BNB_exchange_contract.events, event.event_name)
    # result = event_fn().get_logs(from_block=i, to_block=i + step)
    r = uniswap_v1_BNB_exchange_contract.events.AddLiquidity.get_logs(
        from_block=i, to_block=i + step
    )
    # r == s ? True
    s = uniswap_v1_BNB_exchange_contract.events.AddLiquidity().get_logs(
        from_block=i, to_block=i + step
    )
    t = uniswap_v1_BNB_exchange_contract.events.Transfer.get_logs(
        from_block=i, to_block=i + step
    )
    u = uniswap_v1_BNB_exchange_contract.events.Transfer().get_logs(
        from_block=i, to_block=i + step
    )
    v = uniswap_v1_BNB_exchange_contract.events.AddLiquidity.create_filter(
        from_block=i, to_block=i + step
    )
    w = uniswap_v1_BNB_exchange_contract.events.AddLiquidity().create_filter(
        from_block=i, to_block=i + step
    )
    # result = uniswap_v1_BNB_exchange_contract.events..get_logs(
    #    from_block=i, to_block=i + step
    # )
    for event in v.get_new_entries():
        print("Event:", event["event"])
        print("Args:", event["args"])
        print("Transaction hash:", event["transactionHash"].hex())
        print("Block number:", event["blockNumber"])
        print("---")
    for event in w.get_new_entries():
        print("Event:", event["event"])
        print("Args:", event["args"])
        print("Transaction hash:", event["transactionHash"].hex())
        print("Block number:", event["blockNumber"])
        print("---")
    print(f"{(r, s, t, u, v, w)}")
    print(f"{i} - {i+step}")

    break


([AttributeDict({'args': AttributeDict({'provider': '0x3DE8C28084fd46F3c47D2bA16784a95E647f25B6', 'eth_amount': 1000000000000000000, 'token_amount': 20200000000000000000}), 'event': 'AddLiquidity', 'logIndex': 40, 'transactionIndex': 75, 'transactionHash': HexBytes('0x1b53439a36b357c712a4abe860607c6e4d88a002dd26f97244a8ef3208b2f8b6'), 'address': '0x255e60c9d597dCAA66006A904eD36424F7B26286', 'blockHash': HexBytes('0x96ba61eb50602507ce0bcfc98ec4e9b363d624a6efb1140916be38906fcdad66'), 'blockNumber': 6845141}), AttributeDict({'args': AttributeDict({'provider': '0x3DE8C28084fd46F3c47D2bA16784a95E647f25B6', 'eth_amount': 2500000000000000000, 'token_amount': 50500000000000000001}), 'event': 'AddLiquidity', 'logIndex': 15, 'transactionIndex': 23, 'transactionHash': HexBytes('0x883fa20677e652038a11297b3cc33b3ff6b694fb6fe7cbc1cbbec0117669387b'), 'address': '0x255e60c9d597dCAA66006A904eD36424F7B26286', 'blockHash': HexBytes('0xa9c31a421bb4ac35b46ae2b50c65a48b4c493cde42b74f2c8e9de5f3bfcd7057'), 'b

In [ ]:
# Super important code, for 1 transaction, we get all the Transfer event and we analyze which token has been exchanged
# we also get the Gas + ETH send. If we analyze all the transaction from 1 exchange, we can probably deduct all the liquidity
# token issued by the pair_exchange
from web3.logs import STRICT, IGNORE, DISCARD, WARN

transaction = w3.eth.get_transaction(
    "1b53439a36b357c712a4abe860607c6e4d88a002dd26f97244a8ef3208b2f8b6"
)
d_transaction = event_to_dict(transaction)
tx_eth_value = w3.from_wei(d_transaction["value"], "ether")
decoded = uniswap_v1_BNB_exchange_contract.events.Transfer().process_receipt(
    w3.eth.get_transaction_receipt(transaction.hash),
    DISCARD,
)
for ev in decoded:
    d_ev = event_to_dict(ev)
    d_from = d_ev['args']['_from']
    d_to = d_ev['args']['_to']
    d_value = w3.from_wei(d_ev["args"]["_value"], "ether")
    d_address = d_ev['address']
    d_block = d_ev['blockNumber']
    d_tx_hash = d_ev["transactionHash"]
    token_address = w3.to_checksum_address(d_address)
    token_abi = get_abi(token_address, ETHERSCAN_API_KEY)
    token_contract = w3.eth.contract(address=token_address, abi=token_abi)
    symbol = token_contract.functions.symbol().call()
    decimals = token_contract.functions.decimals().call()
    print(
        f"Block number {d_block}, {tx_eth_value} ETH was used, {d_to} received {d_value} of {symbol} from {d_from} (tx_hash is {d_tx_hash})"
    )

Block number 6845141, 1 ETH was used, 0x255e60c9d597dCAA66006A904eD36424F7B26286 received 20.2 of BNB from 0x3DE8C28084fd46F3c47D2bA16784a95E647f25B6 (tx_hash is 1b53439a36b357c712a4abe860607c6e4d88a002dd26f97244a8ef3208b2f8b6)
Block number 6845141, 1 ETH was used, 0x3DE8C28084fd46F3c47D2bA16784a95E647f25B6 received 1 of b'UNI-V1\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00' from 0x0000000000000000000000000000000000000000 (tx_hash is 1b53439a36b357c712a4abe860607c6e4d88a002dd26f97244a8ef3208b2f8b6)


In [ ]:
# Copy of the exploration block to trace function block by block
# Super interesting to get all the liquidity
# The problem is, we need the receipt of the transaction, and also its not straightforward to see
# the amount of liquidity directly from those call
# 

uniswap_v1_factory_address = w3.to_checksum_address(
    "0x255e60c9d597dCAA66006A904eD36424F7B26286"
    #"0x2157A7894439191E520825FE9399AB8655E0F708"
)
uniswap_v1_factory_abi = get_abi(uniswap_v1_factory_address, ETHERSCAN_API_KEY)
uniswap_v1_factory_contract = w3.eth.contract(
    address=uniswap_v1_factory_address, abi=uniswap_v1_factory_abi
)

def get_internal_transactions_for_contract(
    contract_address: str, from_block: int, to_block: int
):
    """Scan blocks, identify txs to/from contract, and trace internal calls."""
    results = []
    for block_num in range(from_block, to_block + 1):
        block = w3.eth.get_block(block_num, full_transactions=True)
        for tx in block.transactions:
            if (
                tx["to"]
                and (tx["to"] == uniswap_v1_factory_address
                or tx["from"] == uniswap_v1_factory_address)
            ):
                function, params = uniswap_v1_factory_contract.decode_function_input(
                    tx["input"]
                )
                if function.fn_name == "addLiquidity":
                    print(tx)
                    print('Called function:', function.fn_name)
                    print('With arguments:', params)
                    # print(f"Deadline: {block_to_utc(params['deadline'])}")
                    print(f"Deadline: {datetime.fromtimestamp(params['deadline'], tz=timezone.utc)}")
                    print(f"Value: {w3.from_wei(tx.value, 'ether')}")
                    try:
                        print(f"Receipt from transaction: {w3.eth.get_transaction_receipt(tx.hash)}")
                    except:
                        print(f"Can't find 0x{tx.hash.hex()}")
                    # tx.hash.hex()
    return results

# Interesting but I'm pruned
# trace = w3.provider.make_request("trace_transaction", [tx_hash])
# print(trace)
block_1 = 6845140
block_2 = 6850000
internal_txs = get_internal_transactions_for_contract(
    uniswap_v1_factory_contract,
    block_1,
    block_2
    #w3.eth.get_block("latest").number,
)

AttributeDict({'type': 0, 'chainId': 1, 'nonce': 2, 'gasPrice': 5000000000, 'gas': 165556, 'to': '0x255e60c9d597dCAA66006A904eD36424F7B26286', 'value': 1000000000000000000, 'input': HexBytes('0x422f104300000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000011854d0f9cee40000000000000000000000000000000000000000000000000000000000005c0af767'), 'r': HexBytes('0x548fe1148970386fdfaa48d929aa23478303a8f7d17f5fc337e8de1f3b64d51f'), 's': HexBytes('0x44135f14d5ed009876e4121fc7e86d349c73332ebcab17787fa1cc3acf8b3899'), 'v': 37, 'hash': HexBytes('0x1b53439a36b357c712a4abe860607c6e4d88a002dd26f97244a8ef3208b2f8b6'), 'blockHash': HexBytes('0x96ba61eb50602507ce0bcfc98ec4e9b363d624a6efb1140916be38906fcdad66'), 'blockNumber': 6845141, 'transactionIndex': 75, 'from': '0x3DE8C28084fd46F3c47D2bA16784a95E647f25B6'})
Called function: addLiquidity
With arguments: {'min_liquidity': 0, 'max_tokens': 20200000000000000000, 'deadline': 1544222567}
Deadline: 20

In [ ]:
# Very old piece of code, just interesting for getting the signature of event
# Let's keep it in case of

# Example: get the Transfer event signature.
transfer_sig = get_event_signature("Transfer", token_abi)
add_liq_sig = get_event_signature("AddLiquidity", token_abi)
remove_liq_sig = get_event_signature("RemoveLiquidity", token_abi)
print("Transfer signature hash:", transfer_sig)

# -- Step 3: Determine Token Genesis Block and Set Starting Block --
# Assume you have a helper function get_contract_creation_block() that returns the creation block number.
try:
    genesis_block = get_contract_creation_block(token_address, ETHERSCAN_API_KEY)
    start_block = max(genesis_block - 1, 0)
except Exception as e:
    print("Error retrieving genesis block, defaulting to block 0:", e)
    start_block = 0

# -- Step 4: Fetch Transfer Events and Dump to a File (JSON Serializing) --
def get_transfer_events_paginated(token_contract, from_block: int, to_block: int, chunk_size: int = 5000, max_workers: int = 1) -> list:
    """
    Fetches Transfer events for a token_contract in the block range [from_block, to_block],
    paginating by chunk_size to avoid Infura's result limit. Uses moderate parallelization.

    Args:
        token_contract: A Web3 contract instance with a loaded ABI.
        from_block (int): The starting block number.
        to_block (int): The ending block number.
        chunk_size (int): How many blocks to query per chunk (default 5000).
        max_workers (int): Maximum number of parallel workers (default 4).

    Returns:
        List of events.
    """
    events_collected = []
    block_ranges = []
    
    # Divide the full range into chunks.
    for start_blk in range(from_block, to_block + 1, chunk_size):
        end_blk = min(start_blk + chunk_size - 1, to_block)
        block_ranges.append((start_blk, end_blk))
    
    def fetch_range(brange):
        print(f"Fetching for {brange}")
        start_blk, end_blk = brange
        attempts = 0
        max_retries = 5
        while attempts < max_retries:
            try:
                # Add delay to mitigate rate limits.
                time.sleep(random.uniform(1, 3))
                #events = token_contract.events.Transfer.get_logs(from_block=start_blk, to_block=end_blk)
                events = token_contract.events.AddLiquidity.get_logs(from_block=start_blk, to_block=end_blk)
                print(len(events))
                return events
            except Exception as e:
                if "429" in str(e):
                    sleep_time = random.uniform(1, 5)
                    print(f"429 error for blocks {start_blk}-{end_blk}: retrying after {sleep_time:.2f} seconds...")
                    time.sleep(sleep_time)
                    attempts += 1
                else:
                    print(f"Error fetching logs for blocks {start_blk}-{end_blk}: {e}")
                    return []
        return []  # Return empty list if all retries fail.
    
    # Use moderate parallelization.
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_range = {executor.submit(fetch_range, brange): brange for brange in block_ranges}
        for future in concurrent.futures.as_completed(future_to_range):
            events = future.result()
            events_collected.extend(events)
    
    return events_collected
    
# Custom function to convert a web3 event (and its custom types) to a plain dict.
def serialize_event(event):
    # Convert the AttributeDict to a normal dict.
    event_dict = dict(event)
    # Ensure all values are JSON serializable (convert any bytes, HexBytes etc. to a string)
    for key, value in event_dict.items():
        if hasattr(value, "hex"):
            event_dict[key] = value.hex()
    # Also convert inner "args" if present.
    if "args" in event_dict:
        args = dict(event_dict["args"])
        for k, v in args.items():
            if hasattr(v, "hex"):
                args[k] = v.hex()
        event_dict["args"] = args
    return event_dict

# Fetch logs from start_block to the current block (latest)
latest_block = w3.eth.block_number


event_list = get_transfer_events_paginated(token_contract, start_block, latest_block)
# Dump the result to a file (pretty-printing the JSON)
output_filename = f"transfer_events_{contract_address}.json"
with open(output_filename, "w") as f:
    import json
    json.dump(serialized_events, f, indent=4)

print(f"Dumped {len(serialized_events)} events to {output_filename}")


In [ ]:
# Find the amount of token depending on the contract at the very specific block_number
# but it use ETHERSCAN API (to go further: explorer the reconstruct from all the Transfer event but slow)
# Not super useful for the moment

def get_erc20_balance_at_block(contract, address, block_number):
    """
    Query ERC-20 balance of an address at a specific block.

    Parameters:
        contract: Web3 contract instance for the ERC-20 token
        address: string, account to check
        block_number: int, historical block

    Returns:
        int: token balance
    """
    balance = contract.functions.balanceOf(address).call(block_identifier=block_number)
    return balance


# Example usage:
uniswap_v1_BNB_exchange_address = w3.to_checksum_address(
    "0x255e60c9d597dCAA66006A904eD36424F7B26286"
    # "0x2157A7894439191E520825FE9399AB8655E0F708"
)
uniswap_v1_BNB_exchange_abi = get_abi(
    uniswap_v1_BNB_exchange_address, ETHERSCAN_API_KEY
)
uniswap_v1_BNB_exchange_contract = w3.eth.contract(
    address=uniswap_v1_BNB_exchange_address, abi=uniswap_v1_BNB_exchange_abi
)
user_address = "0x3DE8C28084fd46F3c47D2bA16784a95E647f25B6"
block_number = 6845141

balance = get_erc20_balance_at_block(
    uniswap_v1_BNB_exchange_contract, user_address, block_number
)
print(f"Balance at block {block_number}: {balance / 1e18} tokens")

Balance at block 6845141: 1.0 tokens


In [ ]:
# This piece of code explore block 1 by 1 to find every transaction emitted or received from 1 address
# It's super slow but very deep investigation

def get_internal_transactions_with_trace(
    contract_address: str,
    from_block: int,
    to_block: int,
    max_workers: int = 16,
):
    """
    Fetch all transactions involving a contract and optionally trace internal calls.

    Parameters:
        w3: Web3 instance connected to a local archive node.
        contract_address: Ethereum contract address (string).
        from_block: Starting block number (int).
        to_block: Ending block number (int).
        max_workers: Number of threads for parallel fetching.

    Returns:
        List of dictionaries, each containing:
            - 'transaction': The transaction object.
            - 'internal_calls': List of internal calls (empty if trace not available).
    """
    results = []

    def process_block(block_number: int):
        """Fetch transactions in a block and trace internal calls if available."""
        block = w3.eth.get_block(block_number, full_transactions=True)
        block_results = []

        for tx in block.transactions:
            # Filter top-level transactions to/from the contract
            if tx["to"] == contract_address or tx["from"] == contract_address:
                tx_entry = {"transaction": tx, "internal_calls": []}

                # Try to fetch internal calls via trace_transaction
                try:
                    trace = w3.manager.request_blocking(
                        "trace_transaction", [tx["hash"].hex()]
                    )
                    tx_entry["internal_calls"] = trace
                except Exception as e:
                    # If trace API not enabled, just skip internal calls
                    tx_entry["internal_calls"] = []

                block_results.append(tx_entry)

        return block_results

    # Use ThreadPoolExecutor to fetch blocks in parallel
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(process_block, b): b
            for b in range(from_block, to_block + 1)
        }
        for future in as_completed(futures):
            results.extend(future.result())

    return results

block_1 = 6845140
block_2 = 6850000
contract_address = "0x255e60c9d597dCAA66006A904eD36424F7B26286"
txs_with_internal = get_internal_transactions_with_trace(contract_address, block_1, block_2)
for tx_entry in txs_with_internal:
    print(tx_entry)

[{'transaction': AttributeDict({'type': 0, 'chainId': 1, 'nonce': 2, 'gasPrice': 5000000000, 'gas': 165556, 'to': '0x255e60c9d597dCAA66006A904eD36424F7B26286', 'value': 1000000000000000000, 'input': HexBytes('0x422f104300000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000011854d0f9cee40000000000000000000000000000000000000000000000000000000000005c0af767'), 'r': HexBytes('0x548fe1148970386fdfaa48d929aa23478303a8f7d17f5fc337e8de1f3b64d51f'), 's': HexBytes('0x44135f14d5ed009876e4121fc7e86d349c73332ebcab17787fa1cc3acf8b3899'), 'v': 37, 'hash': HexBytes('0x1b53439a36b357c712a4abe860607c6e4d88a002dd26f97244a8ef3208b2f8b6'), 'blockHash': HexBytes('0x96ba61eb50602507ce0bcfc98ec4e9b363d624a6efb1140916be38906fcdad66'), 'blockNumber': 6845141, 'transactionIndex': 75, 'from': '0x3DE8C28084fd46F3c47D2bA16784a95E647f25B6'}), 'internal_calls': []}, {'transaction': AttributeDict({'type': 0, 'chainId': 1, 'nonce': 3, 'gasPrice': 10000000000, 'gas'